## Gold Layer: Historical Product Dimension (SCD Type 2)
In this stage, we initialize our Product Dimension table. We use specific Delta properties to ensure we maintain 30 days of log history, allowing for efficient time travel and auditability of our product catalog.

In [0]:
%run "../00_Setup_Config/project_config"

In [0]:
import dlt
from pyspark.sql.functions import col

# 1. Create the target table first (Required for apply_changes)
dlt.create_streaming_table(
    name="dim_products_scd2",
    comment="SCD Type 2 Product Dimension",
    table_properties={"delta.logRetentionDuration": "interval 30 days"}
)

# 2. Apply SCD Type 2 Logic
dlt.apply_changes(
    target = "dim_products_scd2",
    source = "events_cleaned",
    keys = ["product_id"],      # Unique identifier
    sequence_by = col("event_time"), # Ensures we process updates in order
    stored_as_scd_type = "2",   # <--- This is the magic for Type 2
    track_history_column_list = ["price", "category_code", "brand"]
)